# 03 - Model Prototyping and Backtest from Processed Files
This notebook loads model-ready processed data from data/processed and runs walk-forward modeling plus portfolio diagnostics.

In [ ]:
# pyright: reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false, reportUnknownLambdaType=false, reportUnknownParameterType=false
from pathlib import Path
import logging
import sys
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.figure import Figure

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.backtester import (
    apply_risk_constraints,
    assign_portfolio_weights,
    compute_daily_strategy_returns_realistic,
    summarize_performance,
)
from src.config import get_settings
from src.dashboard import build_concise_dashboard
from src.experiment_tracking import log_mlflow_run
from src.logging_utils import configure_logging
from src.models_ml import ModelSpec, ValidationSpec, walk_forward_predictions

configure_logging(logging.INFO)
logger = logging.getLogger("notebook.03_model_prototyping")
logger.info("Notebook initialization complete.")

In [ ]:
logger.info("Step 1/5: Loading model-ready processed dataset from data/processed.")
settings = get_settings()

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_parquet_path = processed_dir / "model_dataset.parquet"
processed_csv_path = processed_dir / "model_dataset.csv"

dataset: pd.DataFrame | None = None
source_path: Path | None = None
if processed_parquet_path.exists():
    try:
        dataset = pd.read_parquet(processed_parquet_path)
        source_path = processed_parquet_path
    except ImportError:
        logger.warning("Parquet engine not available; falling back to CSV processed data.")

if dataset is None and processed_csv_path.exists():
    dataset = pd.read_csv(processed_csv_path)
    source_path = processed_csv_path

if dataset is None or source_path is None:
    raise FileNotFoundError(
        f"Processed dataset not found. Expected one of: {processed_parquet_path} or {processed_csv_path}. "
        "Run notebook 02 first."
    )

logger.info("Loaded processed dataset rows: %d from %s", len(dataset), source_path)
print("Processed source:", source_path)
print("Model dataset rows:", len(dataset))

In [ ]:
logger.info("Step 4/5: Training walk-forward model and generating OOS predictions.")
assert dataset is not None
spec = ModelSpec(name="random_forest", random_state=settings.random_state)
validation = ValidationSpec(n_splits=5, purge_days=5, embargo_days=2, min_train_dates=120)

pred_df = walk_forward_predictions(dataset, spec=spec, validation=validation)
logger.info("Predictions generated: %d rows.", len(pred_df))
print("Out-of-sample predictions:", len(pred_df))
pred_df.head()

In [ ]:
logger.info("Step 5/5: Converting predictions to risk-constrained, realistic net returns.")
weighted_raw = assign_portfolio_weights(pred_df, quantile=settings.top_bottom_quantile)
weighted = apply_risk_constraints(weighted_raw, max_abs_weight=0.25, max_gross_leverage=1.0)

daily = compute_daily_strategy_returns_realistic(
    weighted,
    transaction_cost_bps=5.0,
    slippage_bps=2.0,
)
perf = summarize_performance(daily[['date', 'strategy_return']])
logger.info("Performance dictionary computed successfully.")
perf

In [ ]:
# pyright: reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false, reportUnknownLambdaType=false, reportUnknownParameterType=false
fig, ax = plt.subplots(figsize=(12, 4))
daily.set_index('date')['strategy_return'].cumsum().plot(ax=ax, title='Cumulative Strategy Return')
ax.set_ylabel('Cumulative Return')
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# pyright: reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false, reportUnknownLambdaType=false, reportUnknownParameterType=false
logger.info("Rendering extended diagnostics: drawdown, rolling Sharpe, return distribution, and cross-sectional signal quality.")

daily_plot = daily.copy()
daily_plot['date'] = pd.to_datetime(daily_plot['date'])
daily_plot = daily_plot.sort_values('date').reset_index(drop=True)

equity = (1.0 + daily_plot['strategy_return']).cumprod()
running_peak = equity.cummax()
drawdown = (equity / running_peak) - 1.0

rolling_mean = daily_plot['strategy_return'].rolling(63).mean()
rolling_std = daily_plot['strategy_return'].rolling(63).std(ddof=1)
rolling_sharpe = (rolling_mean / rolling_std) * (252 ** 0.5)

pred_plot = pred_df.copy()
pred_plot['date'] = pd.to_datetime(pred_plot['date'])
ic_by_day = (
    pred_plot.groupby('date')[['prediction', 'target_fwd_return']]
    .corr(method='spearman')
    .reset_index()
    .query("level_1 == 'prediction'")
    .set_index('date')['target_fwd_return']
    .rename('ic')
    .dropna()
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(daily_plot['date'], drawdown, color='firebrick')
axes[0, 0].set_title('Strategy Drawdown')
axes[0, 0].set_ylabel('Drawdown')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(daily_plot['date'], rolling_sharpe, color='darkgreen')
axes[0, 1].axhline(0.0, linestyle='--', color='black', alpha=0.6)
axes[0, 1].set_title('63-Day Rolling Sharpe')
axes[0, 1].set_ylabel('Sharpe')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].hist(daily_plot['strategy_return'], bins=40, color='steelblue', alpha=0.85)
axes[1, 0].set_title('Daily Net Return Distribution')
axes[1, 0].set_xlabel('Daily Return')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(ic_by_day.index, ic_by_day.values, color='purple', alpha=0.8, label='Daily IC')
axes[1, 1].axhline(ic_by_day.mean(), color='black', linestyle='--', label=f"Mean IC: {ic_by_day.mean():.3f}")
axes[1, 1].set_title('Daily Spearman IC')
axes[1, 1].set_ylabel('IC')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

long_short_counts = (
    weighted.groupby(['date', 'side']).size().unstack(fill_value=0).sort_index()
    if not weighted.empty else pd.DataFrame()
)
if not long_short_counts.empty:
    long_short_counts.index = pd.to_datetime(long_short_counts.index)
    ax = long_short_counts.plot(kind='area', stacked=True, figsize=(12, 4), alpha=0.7)
    ax.set_title('Daily Position Count by Side')
    ax.set_ylabel('Count')
    ax.grid(alpha=0.3)
    plt.show()

In [ ]:
# pyright: reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false, reportUnknownLambdaType=false, reportUnknownParameterType=false
output_dir = PROJECT_ROOT / "docs" / "figures"
output_dir.mkdir(parents=True, exist_ok=True)

# 1) Cumulative return
fig, ax = plt.subplots(figsize=(12, 4))
daily.set_index("date")["strategy_return"].cumsum().plot(ax=ax, title="Cumulative Strategy Return (Net)")
ax.set_ylabel("Cumulative Return")
ax.grid(alpha=0.3)
cum_path = output_dir / "cumulative_strategy_return.png"
fig.savefig(cum_path, dpi=180, bbox_inches="tight")
plt.close(fig)

# 2) 2x2 diagnostics
daily_plot = daily.copy()
daily_plot["date"] = pd.to_datetime(daily_plot["date"])
daily_plot = daily_plot.sort_values("date").reset_index(drop=True)

equity = (1.0 + daily_plot["strategy_return"]).cumprod()
running_peak = equity.cummax()
drawdown = (equity / running_peak) - 1.0

rolling_mean = daily_plot["strategy_return"].rolling(63).mean()
rolling_std = daily_plot["strategy_return"].rolling(63).std(ddof=1)
rolling_sharpe = (rolling_mean / rolling_std) * (252 ** 0.5)

pred_plot = pred_df.copy()
pred_plot["date"] = pd.to_datetime(pred_plot["date"])
ic_by_day = (
    pred_plot.groupby("date")[["prediction", "target_fwd_return"]]
    .corr(method="spearman")
    .reset_index()
    .query("level_1 == 'prediction'")
    .set_index("date")["target_fwd_return"]
    .rename("ic")
    .dropna()
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes[0, 0].plot(daily_plot["date"], drawdown, color="firebrick")
axes[0, 0].set_title("Strategy Drawdown")
axes[0, 0].set_ylabel("Drawdown")
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(daily_plot["date"], rolling_sharpe, color="darkgreen")
axes[0, 1].axhline(0.0, linestyle="--", color="black", alpha=0.6)
axes[0, 1].set_title("63-Day Rolling Sharpe")
axes[0, 1].set_ylabel("Sharpe")
axes[0, 1].grid(alpha=0.3)

axes[1, 0].hist(daily_plot["strategy_return"], bins=40, color="steelblue", alpha=0.85)
axes[1, 0].set_title("Daily Net Return Distribution")
axes[1, 0].set_xlabel("Daily Return")
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(ic_by_day.index, ic_by_day.values, color="purple", alpha=0.8, label="Daily IC")
axes[1, 1].axhline(ic_by_day.mean(), color="black", linestyle="--", label=f"Mean IC: {ic_by_day.mean():.3f}")
axes[1, 1].set_title("Daily Spearman IC")
axes[1, 1].set_ylabel("IC")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)
plt.tight_layout()
diag_path = output_dir / "diagnostics_grid.png"
fig.savefig(diag_path, dpi=180, bbox_inches="tight")
plt.close(fig)

# 3) Daily position counts by side
long_short_counts = weighted.groupby(["date", "side"]).size().unstack(fill_value=0).sort_index()
long_short_counts.index = pd.to_datetime(long_short_counts.index)
ax = long_short_counts.plot(kind="area", stacked=True, figsize=(12, 4), alpha=0.7, title="Daily Position Count by Side")
ax.set_ylabel("Count")
ax.grid(alpha=0.3)
pos_fig = ax.get_figure()
if isinstance(pos_fig, Figure):
    pos_path = output_dir / "position_count_by_side.png"
    pos_fig.savefig(pos_path, dpi=180, bbox_inches="tight")
    plt.close(pos_fig)
else:
    pos_path = output_dir / "position_count_by_side.png"

# 4) Concise dashboard
dash_path = output_dir / "concise_dashboard.png"
_ = build_concise_dashboard(
    daily_returns=daily[["date", "strategy_return"]],
    perf=perf,
    ic_by_day=ic_by_day,
    output_path=dash_path,
)
plt.close("all")

# 5) MLflow tracking
params: dict[str, Any] = {
    "model_name": spec.name,
    "random_state": spec.random_state,
    "n_splits": validation.n_splits,
    "purge_days": validation.purge_days,
    "embargo_days": validation.embargo_days,
    "quantile": settings.top_bottom_quantile,
    "transaction_cost_bps": 5.0,
    "slippage_bps": 2.0,
}
run_logged = log_mlflow_run(
    experiment_name="cross_sectional_alpha",
    run_name="rf_walkforward_research_grade",
    params=params,
    metrics=perf,
    artifact_paths=[cum_path, diag_path, pos_path, dash_path],
)
print("MLflow run logged:", run_logged)

print("Saved figures to:", output_dir)
sorted([p.name for p in output_dir.glob("*.png")])